In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
import torch
from torch import optim
import time
from time import time
import random
import sys
from pathlib import Path
current_dir = Path().resolve()  
ModelBase_path = current_dir.parent/'Model_Base'
sys.path.append(str(ModelBase_path))
from Model import ResNetGELU
print(ModelBase_path)

/home/xuty/Xutycode/RSV4HQ_260402/Code/Model_Base


In [2]:
def add_velocity_uncertainty(Ux, uncertainty=0.2, method='normal', seed=None):
    """
    为表面速度添加 ±20% 随机不确定性
    - method: 'uniform' 均匀分布 | 'normal' 正态分布（推荐，物理更合理）
    - seed:   随机种子，复现用
    """
    if seed is not None:
        np.random.seed(seed)
    if method == 'normal':
        # 3σ = uncertainty→  σ = uncertainty/3
        noise = np.random.normal(1.0, uncertainty / 3, len(Ux))
    else:
        noise = np.random.uniform(1- uncertainty, 1 + uncertainty, len(Ux))
    return Ux * noise

# ── 2. 平滑函数（三种，可按需选择） ──────────────────────────
def smooth_velocity(Ux_noisy, method='savgol', window=7, polyorder=3, sigma=2):
    """
    对含噪速度进行平滑，减少拐点
    参数:
    - method: 'savgol'→ Savitzky-Golay 局部多项式最小二乘 ✅推荐
                'gaussian' → 高斯核卷积（各向同性平滑）
                  'moving'   → 滑动均值（最简单）
    - window    : 滑动窗口长度（必须为奇数）
    - polyorder : SG滤波多项式阶数（window > polyorder）
    - sigma     : 高斯核标准差（控制平滑程度）
    
    数学原理:
    - Savitzky-Golay: 在每个窗口内用k阶多项式做最小二乘拟合，保留峰值形状，不损失极值
    - Gaussian:y_smooth[i] = Σ G(j)·y[i+j], G~N(0,σ²)
    - Moving Average: y_smooth[i] = mean(y[i-w:i+w])
    """
    if method == 'savgol':
        # window 必须 <= len(Ux_noisy) 且为奇数
        win = min(window, len(Ux_noisy) if len(Ux_noisy) % 2 != 0
                  else len(Ux_noisy) - 1)
        return savgol_filter(Ux_noisy, window_length=win, polyorder=polyorder)

    elif method == 'gaussian':
        return gaussian_filter1d(Ux_noisy, sigma=sigma)

    elif method == 'moving':
        kernel = np.ones(window) / window
        # 'same' 保持长度不变；边缘用镜像延拓
        return np.convolve(
            np.pad(Ux_noisy, window // 2, mode='reflect'),
            kernel, mode='valid'
        )[:len(Ux_noisy)]

    else:
        raise ValueError("method必须为 'savgol' | 'gaussian' | 'moving'")

In [4]:
def load_array(data_arrays, batch_size, is_train=True):  #@save

    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

def fun_calDiff(data,y,step):
    if len(data) != len(y):
        raise ValueError("data and y must have the same length")

    indices = [i * step + int(step/2) for i in range(round(len(data) / step))]
    data = [data[i] for i in indices]
    y = [y[i] for i in indices]

    U_avi,U_diff,Y_diff = [],[],[]

    for i in range(len(data) - 1):
        if y[i+1] == y[i]:
            raise ValueError("y[i+1] - y[i] is zero, cannot divide by zero")
        U_avi.append((data[i+1] + data[i])/2)
        U_diff.append((data[i+1] - data[i]) / (y[i+1] - y[i]))
        Y_diff.append(y[i+1] - y[i])

    return U_avi,U_diff, Y_diff

def splitData0(data0,step):
    U0 = data0[0]
    H0 = data0[1]
    B0 = data0[int(len(data0)-1)]
    Fr = (U0/np.sqrt(9.81*H0))
    X = B0+2*H0
    R=B0*H0/(B0+2*H0)
    Re = U0*R/1e-6
    dsize = int((len(data0)-2)/2.0)
    Ux0 = data0[2:dsize+2]
    Y = data0[dsize+2:2*dsize+2]
    half_dsize = int(dsize/2.0)
    inValueU = Ux0
    # [:half_dsize]
    inValueY = Y[:half_dsize]

## Data Grade Enhance————
    aver = sum(Ux0)/len(Ux0)
    Uamx_Uaver= max(Ux0)/aver
    inValueU = Ux0/np.average(inValueU)
    U_avi,U_diff,Y_diff = fun_calDiff(Ux0,Y,step)
##—————————————————————————————

    pin_np1 = np.array(U_avi).reshape(1,-1)
    pin_np2 = np.array(U_diff).reshape(1,-1)
    pin_np3 = np.array(Y_diff).reshape(1,-1)
    pin_np = np.vstack((pin_np1, pin_np2, pin_np3))
    target_np = np.array([U0,H0])
    Z = np.array([U0,H0,B0,Fr,Re,X,R,step,aver,Uamx_Uaver])
    return pin_np, target_np ,Z

def genFeaturesLabels(data_set,step=5,maxh=10,maxFr=0.99,maxHoverB=1.2):
    features,labels,z=[],[],[]
    for data0 in data_set:
        pin_np, target_np ,zi=splitData0(data0,step)
        if zi[9]>1.1:
            continue
        if zi[1]>maxh:
            continue
        if zi[3]>maxFr:
            continue
        if np.log(zi[1]/zi[2])>maxHoverB:
            continue
        features.append(pin_np)
        labels.append(target_np)
        z.append(zi)
    features =  torch.from_numpy(np.array(features))
    labels =  torch.from_numpy(np.array(labels))
    z = torch.from_numpy(np.array(z))
    return features, labels ,z

In [5]:
def test_loss(NAME,test_iter,cuda=False):
    # read model
    PATH = NAME
    if(cuda):
        net = torch.load(PATH,weights_only=False)
    else:
        net = torch.load(PATH,map_location='cpu',weights_only=False )
    net.eval()
    if cuda:
        net.cuda()
    else:
        net.cpu()    

    # read data set

    test_loss = []

    for X,y,z in test_iter:
        pin_np, target_np ,z= X,y,z
        pin = torch.from_numpy(np.float32(pin_np))
        pin.requires_grad = False
        if cuda:
            pred = net(pin.cuda())
        else:
            pred = net(pin.cpu()) 
        #print(pred.detach().cpu().numpy())
        UH_preds = pred.detach().cpu().numpy()
        for UH_pred,pnp,tnp in zip(UH_preds,pin_np,target_np):
            U0_pred = UH_pred[0]
            H0_pred = UH_pred[1]
            loss0 = (((U0_pred-tnp[0])/tnp[0])**2+((H0_pred-tnp[1])/tnp[1])**2)**0.5
            test_loss.append(loss0)
    return np.mean(test_loss)   

def train(NAME,net,reload=False,LR = 0.0001,n_epochs=200,batch_size=30,cuda=False,step=5,maxh=10,maxFr=0.99,maxHoverB=1.2):
    if(reload):
        PATH = NAME
        if(cuda):
            net = torch.load(PATH,weights_only=False )
        else:
            net = torch.load(PATH,map_location='cpu',weights_only=False )
        net.eval()        
    if cuda:
        net.cuda()
    else:
        net.cpu()

    # read data set
    global data_set
    data_size=len(data_set)
    train_size = int(data_size*0.8)
    random.seed(1)
    train_cases = random.sample(list(np.arange(0,data_size)), train_size)
    train_cases.sort()
    test_cases = []
    for i in range(data_size):
        if i not in train_cases:
            test_cases.append(i)
            
    train_iter = load_array((genFeaturesLabels(data_set[train_cases],step,maxh,maxFr,maxHoverB)), batch_size)
    test_iter = load_array((genFeaturesLabels(data_set[test_cases],step,maxh,maxFr,maxHoverB)), batch_size)
    
    # loss function
    loss_fn = torch.nn.MSELoss(reduction='mean')
    
    optimizer = optim.Adam(net.parameters(), lr=LR)
    epoch_loss = []
    lossLog = open(NAME+"_loss.log", "a")
    if not reload:
        lossLog.close()
        lossLog = open(NAME+"_loss.log", "w")

    for epoch in range(n_epochs):
        for param in net.parameters():
            param.requires_grad = True
        optimizer.zero_grad()

        t0 = time()
        i=0
        for X, y ,z in train_iter:
            #print(epoch,i,X.shape)
            z = z
            i=i+1
            train_loss = []
            pin = torch.from_numpy(np.float32(X))
            pin.requires_grad = True
            if cuda:
                pred = net(pin.cuda())
            else:
                pred = net(pin.cpu())
            #print(pred.shape,y.shape)
            target = y
            if cuda:
                target = target.cuda()
            else:
                target = target.cpu()
            loss = loss_fn(pred,target)
            loss.backward()
            train_loss.append(loss.cpu().detach().numpy())
            for param in net.parameters():
                param.requires_grad = True
            optimizer.step()
            optimizer.zero_grad()
            
        if(len(epoch_loss)>1):
            if(epoch_loss[-1]>epoch_loss[-2]):
                #LR=LR*0.5
                optimizer = optim.Adam(net.parameters(), lr=LR) 
        epoch_loss.append(np.mean(train_loss))
               
        PATH = NAME
        torch.save(net, PATH)
        t_loss = test_loss(NAME,test_iter,cuda)
        #t_loss = 0
        if epoch % 10 ==0: 
            print(epoch)
            print(epoch,epoch_loss[-1],t_loss,LR)
            print("%2.1f s"%(time()-t0))
        lossLog.write("%d %4.5e %4.5e\n"%(epoch,epoch_loss[-1],t_loss))
        lossLog.close()
        lossLog = open(NAME+"_loss.log", "a")        
    lossLog.close()   
    PATH = NAME
    torch.save(net, PATH)

In [7]:
def trainpro(trainName,net,step=5,maxh=10,maxFr=0.99,maxHoverB=1.2):
    t0 = time()
    reload = True
    scale = 1
    LR = 1.0e-4
    n_epochs = 200
    batch_size = 20
    train(trainName,net,False,LR,n_epochs,batch_size,True,step,maxh,maxFr,maxHoverB)
    LR = 1.0e-4
    n_epochs =200
    batch_size = 20
    train(trainName,net,reload,LR,n_epochs,batch_size,True,step,maxh,maxFr,maxHoverB)
    LR = 1.0e-5
    n_epochs = 200
    batch_size = 20
    train(trainName,net,reload,LR,n_epochs,batch_size,True,step,maxh,maxFr,maxHoverB)
    LR = 1.0e-6
    n_epochs = 200
    batch_size = 20
    train(trainName,net,reload,LR,n_epochs,batch_size,True,step,maxh,maxFr,maxHoverB)
    LR = 1.0e-7
    n_epochs = 200
    batch_size = 20
    train(trainName,net,reload,LR,n_epochs,batch_size,True,step,maxh,maxFr,maxHoverB)
    t1 = time()
    print (t1-t0)

In [ ]:
import os
Data_path = current_dir.parent.parent/'Data/datasets/'
#file_name = 'RiverCalder500.dat'
# file_name = 'case240913_5K.dat'
file_name = 'Random5000.dat'
file_path = os.path.join(Data_path, file_name)
data_set=np.float32(np.loadtxt(file_path,unpack=True))
features, labels ,z = genFeaturesLabels(data_set,step=20,maxFr=0.99,maxHoverB=1)
a=len(features[0,0,:])
b=len(features[0])
net=ResNetGELU(in_channels=b)
trainName="ResNetdatapro3S_20_hoverb1"
trainpro(trainName,net,step=20,maxh=10,maxFr=0.99,maxHoverB=1)

In [ ]:
import os
Data_path = current_dir.parent.parent/'Data/datasets/'
#file_name = 'RiverCalder500.dat'
# file_name = 'case240913_5K.dat'
file_name = 'Random5000.dat'
file_path = os.path.join(Data_path, file_name)
data_set=np.float32(np.loadtxt(file_path,unpack=True))
features, labels ,z = genFeaturesLabels(data_set,step=20,maxFr=0.99,maxHoverB=0.5)
a=len(features[0,0,:])
b=len(features[0])
net=ResNetGELU(in_channels=b)
trainName="ResNetdatapro3S_20_hoverb0.5"
trainpro(trainName,net,step=20,maxh=10,maxFr=0.99,maxHoverB=0.5)

In [ ]:
import os
Data_path = current_dir.parent.parent/'Data/datasets/'
#file_name = 'RiverCalder500.dat'
# file_name = 'case240913_5K.dat'
file_name = 'Random5000.dat'
file_path = os.path.join(Data_path, file_name)
data_set=np.float32(np.loadtxt(file_path,unpack=True))
features, labels ,z = genFeaturesLabels(data_set,step=20,maxFr=0.99,maxHoverB=0)
a=len(features[0,0,:])
b=len(features[0])
net=ResNetGELU(in_channels=b)
trainName="ResNetdatapro3S_20_hoverb0"
trainpro(trainName,net,step=20,maxh=10,maxFr=0.99,maxHoverB=0)